# 🎬 AI 数字人口播视频生成
**EchoMimic V2 + Edge-TTS**

使用方法：点击上方菜单 `Run → Run All Cells`，等待完成即可。

---
**⚠️ 上传照片**：把 `2.png` 拖到左侧文件栏的 `EchoMimic/` 目录里

In [ ]:
# ============================================
# 第一步：检查环境
# ============================================
import os
import sys

print("🐍 Python:", sys.version)
print("📁 当前目录:", os.getcwd())

# 确认关键文件都在
checks = {
    "照片 2.png": "2.png",
    "推理脚本": "infer_acc.py",
    "配置文件": "configs/echomimiv2.yaml",
    "模型权重": "pretrained_weights",
}
all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    print(f"{'✅' if ok else '❌'} {name}: {path}")
    if not ok:
        all_ok = False

if not all_ok:
    print("\n⚠️ 上面标 ❌ 的文件缺失，请检查后重试！")
else:
    print("\n✅ 所有文件就绪！")

In [ ]:
# ============================================
# 第一步附：查看显存
# ============================================
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# ============================================
# 第二步：安装 Edge-TTS（生成语音）
# ============================================
!pip install edge-tts -q
print("✅ Edge-TTS 安装完成")

In [ ]:
# ============================================
# 第三步：用文字生成音频
# 修改 text 变量即可换口播文案
# ============================================
text = "大家好，我是AI数字人，这是用一张照片加上一段文字自动生成的视频，是不是很神奇？科技让创作变得更简单了。"

print(f"📝 口播文案: {text}")
print(f"📝 文案字数: {len(text)} 字")

# 生成音频
!edge-tts --text "{text}" --voice zh-CN-XiaoxiaoNeural --write-media audio.wav

# 检查
if os.path.exists("audio.wav"):
    size_kb = os.path.getsize("audio.wav") / 1024
    print(f"✅ 音频已生成: {size_kb:.1f} KB")
else:
    print("❌ 音频生成失败！")

In [ ]:
# ============================================
# 第四步：EchoMimic V2 推理
# 12GB 显存安全参数：length=60, cfg=1.5
# 生成时间约 5-10 分钟，请耐心等待
# ============================================
print("🎬 开始生成数字人视频...")
print("⏳ 预计需要 5-10 分钟，请耐心等待...")

!python infer_acc.py \
  --config configs/echomimiv2.yaml \
  --source_image 2.png \
  --driven_audio audio.wav \
  --length 60 \
  --cfg 1.5

print("\n✅ 推理完成！")

In [ ]:
# ============================================
# 第五步：找到生成的视频
# ============================================
import glob

# EchoMimic 输出通常在 output/ 或当前目录
videos = glob.glob("*.mp4") + glob.glob("output/**/*.mp4", recursive=True) + glob.glob("results/**/*.mp4", recursive=True)

if videos:
    print("🎬 找到以下视频文件:")
    for v in videos:
        size_mb = os.path.getsize(v) / 1024 / 1024
        print(f"   📹 {v} ({size_mb:.1f} MB)")
    print("\n💡 在左侧文件栏右键视频文件 → Download 即可下载到本地")
else:
    # 可能命名不同
    print("🔍 搜索所有新生成的文件...")
    import time
    for f in os.listdir("."):
        if os.path.isfile(f):
            mtime = os.path.getmtime(f)
            if time.time() - mtime < 600:  # 10分钟内
                size_mb = os.path.getsize(f) / 1024 / 1024
                print(f"   📄 {f} ({size_mb:.1f} MB)")
    print("\n如果没看到视频文件，检查上面推理那一步是否报错")

---
## ✅ 完成！

如果上面推理没报错，视频已经生成好了。

**下载方法**：在 JupyterLab 左侧文件栏里找到 `.mp4` 文件，右键 → **Download**。

---

### 如果爆显存（OOM）

把第四步的命令改成 `--length 40` 再试：

```bash
python infer_acc.py --config configs/echomimiv2.yaml --source_image 2.png --driven_audio audio.wav --length 40 --cfg 1.5
```

### 换口播文案

修改第三步里的 `text = "..."` 变量，再重新运行第三步和第四步。